In [ ]:
import pandas as pd
import numpy as np
import codecademylib3
import matplotlib.pyplot as plt
import seaborn as sns

#Import models from scikit learn module:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, RandomForestRegressor
from sklearn import tree
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

col_names = ['age', 'workclass', 'fnlwgt','education', 'education-num', 
'marital-status', 'occupation', 'relationship', 'race', 'sex',
'capital-gain','capital-loss', 'hours-per-week','native-country', 'income']
df = pd.read_csv('adult.data', header=None, names = col_names)

#Distribution of income
income_distribution = df['income'].value_counts(normalize=True)
#print(income_distribution)

#Clean columns by stripping extra whitespace for columns of type "object"
for c in df.select_dtypes(include=['object']).columns:
  df[c] = df[c].str.strip()
  #print(df[c].head())

feature_cols = ['age',
       'capital-gain', 'capital-loss', 'hours-per-week', 'sex','race']
#Create feature dataframe X with feature columns and dummy variables for categorical features
X = pd.get_dummies(df[feature_cols], drop_first=True)
#Create output variable y which is binary, 0 when income is less than 50k, 1 when it is greather than 50k
y = (df['income'].apply(lambda x: 0 if x == '<=50K' else 1))

#Split data into a train and test set
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=1, test_size=0.20)

#Instantiate random forest classifier, fit and score with default parameters
rfc = RandomForestClassifier()
rfc.fit(x_train, y_train)
rfc_accuracy = rfc.score(x_test, y_test)
#print(rfc_accuracy.round(4))

#Tune the hyperparameter max_depth over a range from 1-25, save scores for test and train set
np.random.seed(0)
accuracy_train=[]
accuracy_test = []
for i in range(25):
  rf = RandomForestClassifier(max_depth=i+1)
  rf.fit(x_train, y_train)
  accuracy_train.append(accuracy_score(rf.predict(x_train), y_train).round(4))
  accuracy_test.append(accuracy_score(rf.predict(x_test), y_test).round(4))

# print(f'Training Accuracy : {accuracy_train}')
# print(f'Test Accuracy : {accuracy_test}')
    
#Find the best accuracy and at what depth that occurs
largest_accuracy = np.max(accuracy_test)
depth = np.argmax(accuracy_test)

# print('Largest Accuracy:', largest_accuracy, 'Depth Occurred:', depth)

#Plot the accuracy scores for the test and train set over the range of depth values  
max_depth = range(1,26)
plt.plot(max_depth, accuracy_test)
plt.plot(max_depth, accuracy_train)
plt.legend(['test accuracy', 'train accuracy'])
plt.xlabel('max depth')
plt.ylabel('accuracy')
plt.show()

#Save the best random forest model and save the feature importances in a dataframe
best_rf = RandomForestClassifier(max_depth=depth)
best_rf.fit(x_train, y_train)
feature_importance = pd.DataFrame({
  'features': x_train.columns, 
  'importance': best_rf.feature_importances_
  })

feature_importance = feature_importance.sort_values(by='importance', ascending=False)
# print(feature_importance.head())

#Create two new features, based on education and native country
df['education_bin'] = pd.cut(df['education-num'], [0,9,13,16], labels=['HS or less', 'College to Bachelors', 'Masters or more'])

feature_cols = ['age',
        'capital-gain', 'capital-loss', 'hours-per-week', 'sex', 'race','education_bin']
#Use these two new additional features and recreate X and test/train split
X = pd.get_dummies(df[feature_cols], drop_first=True)
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=1, test_size=0.20)

np.random.seed(0)
accuracy_train=[]
accuracy_test = []
depths = range(1,10)
for i in depths:
    rf = RandomForestClassifier(max_depth=i)
    rf.fit(x_train, y_train)
    y_pred = rf.predict(x_test)
    accuracy_test.append(accuracy_score(y_test, rf.predict(x_test)).round(4))
    accuracy_train.append(accuracy_score(y_train, rf.predict(x_train)).round(4))

# print(accuracy_test)
# print(accuracy_train)

#Find the best max depth now with the additional two features
largest_accuracy = np.max(accuracy_test)
max_depth = np.argmax(accuracy_test)
# print('Largest Accuracy with additional features:', largest_accuracy)
# print('Depth Occurred:', depth)

#Save the best model and print the two features with the new feature set
depth = range(1,10)
plt.plot(depth, accuracy_test)
plt.plot(depth, accuracy_train)
plt.legend(['test accuracy', 'train accuracy'])
plt.xlabel('max depth')
plt.ylabel('accuracy')
plt.show()

best_rfc2 = RandomForestClassifier(max_depth=max_depth+1)
best_rfc2.fit(x_train, y_train)
feature_importance = pd.DataFrame({
  'features': x_train.columns, 
  'importance': best_rfc2.feature_importances_
  })
feature_importance = feature_importance.sort_values(by='importance', ascending=False)
print(feature_importance)
